# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 52.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
# <Student to fill this section>
group_name = "Neural Minds"
student_name = "Aryaman Malik"
student_id = "25980501"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section>

### 0.b Import Packages

In [6]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [7]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "product_category.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [8]:
# <Student to fill this section>
# Shape, schema, and the full table (only 4 rows!)
print("Shape:", df.shape)
df.info()
print("\nFull table:")
print(df)

Shape: (4, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_category_id  4 non-null      object
 1   name                 4 non-null      object
dtypes: object(2)
memory usage: 196.0+ bytes

Full table:
                    product_category_id         name
0  6b440a50-1a1b-4d3b-988f-076c498bd092        Bikes
1  65e2841b-c7da-4c4a-9565-cd8cfd9a2f68   Components
2  2664e8d8-c7ce-4a38-aae4-07e0b536bba3     Clothing
3  0a37a9c4-764e-4952-bf4f-d47b38694c4d  Accessories


In [9]:
# Quality checks
print("Missing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Unique category_id: {df['product_category_id'].nunique()}")
print(f"Unique name:        {df['name'].nunique()}")

Missing values:
product_category_id    0
name                   0
dtype: int64

Duplicate rows: 0
Unique category_id: 4
Unique name:        4


In [10]:
# <Student to fill this section>
dataset_insights = """
product_category is the top-level dimension of the product hierarchy. It has just 4 rows and 2 columns: product_category_id (UUID) and name. Data quality is perfect - zero nulls, zero duplicates, all four category_ids and names unique.
The 4 categories are: Bikes, Components, Clothing and Accessories.
While the table itself is trivial, this is arguably the most important categorical grouping for the anomaly-detection use case. As we saw in the subcategory EDA, prices vary by roughly 183x across subcategories - but the variation is structured: subcategories within "Bikes" all sit above USD 500, while subcategories within "Accessories" mostly sit below USD 50. The top-level category captures most of that structural variation in just 4 levels, which is ideal for one-hot encoding and for computing stable baseline statistics (a median computed on hundreds of Bikes lines is far more reliable than a median computed on a single-product subcategory).
"""

In [11]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `\<name` (downstream product coverage)`\>`

In [12]:
# <Student to fill this section>
# How many products and how much revenue does each category represent?
sub_df = pd.read_csv(at.folder_path / "product_sub_category.csv").rename(columns={'name':'subcategory_name'})
prod_df = pd.read_csv(at.folder_path / "product.csv").drop_duplicates('product_id').rename(columns={'name':'product_name'})

cat = df.rename(columns={'name':'category_name'})
chain = sub_df.merge(cat, on='product_category_id').merge(prod_df, on='product_subcategory_id', how='left')

print("Subcategories per category:")
print(cat.merge(sub_df, on='product_category_id').groupby('category_name').size())

print("\nPriced products per category:")
print(chain[chain['list_price']>0].groupby('category_name')['product_id'].nunique())

Subcategories per category:
category_name
Accessories    12
Bikes           3
Clothing        8
Components     14
dtype: int64

Priced products per category:
category_name
Accessories     29
Bikes           97
Clothing        35
Components     134
Name: product_id, dtype: int64


In [13]:
coverage = pd.DataFrame({
    'category': ['Bikes', 'Components', 'Clothing', 'Accessories'],
    'subcategories': [3, 14, 8, 12],
    'priced_products': [97, 134, 35, 29]
})
coverage_long = coverage.melt(id_vars='category', var_name='metric', value_name='count')

chart = alt.Chart(coverage_long).mark_bar().encode(
    x=alt.X('category:N', title='Top-level category', sort=['Bikes','Components','Clothing','Accessories']),
    y=alt.Y('count:Q', title='Count'),
    color=alt.Color('metric:N', title='Metric',
                    scale=alt.Scale(range=['#4C78A8', '#F58518'])),
    xOffset='metric:N',
    tooltip=['category', 'metric', 'count']
).properties(
    title='Subcategories vs priced products per category',
    width=500, height=300
)
chart.display()

alt.Chart(...)

In [14]:
# <Student to fill this section>
feature_1_insights = """
The four categories are deeply imbalanced in their internal composition:
- Bikes: only 3 subcategories (Mountain, Road, Touring Bikes) but 97 priced products - a narrow but deep category.
- Components: 14 subcategories and 134 priced products - the broadest in both dimensions.
- Clothing: 8 subcategories with 35 products.
- Accessories: 12 subcategories but only 29 priced products - many one-product subcategories live here.

What this means for the anomaly model:
- Bikes and Components together account for 231 of the 295 priced products (~78%), so they will dominate the model's training signal.
- The Accessories category has many singleton subcategories (only 1 product), which makes subcategory-level baselines unreliable. Falling back to category-level statistics (29 priced Accessories products) gives a much more stable baseline.
- Each category has a clear and distinct price regime, which we will quantify in the next feature analysis - this is precisely why the top-level category is such a strong feature.
"""

In [15]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `\<price_tier_per_category\>`

In [16]:
# <Student to fill this section>
# Price distribution per category (priced products only, deduplicated)
priced = chain[chain['list_price'] > 0].drop_duplicates('product_id')

cat_price = priced.groupby('category_name')['list_price'].describe().round(2)
print("List-price distribution per category (priced products):")
print(cat_price)

List-price distribution per category (priced products):
               count     mean     std     min     25%      50%      75%  \
category_name                                                             
Accessories     29.0    34.35   37.53    2.29    9.99    24.99    34.99   
Bikes           97.0  1586.74  996.27  539.99  769.49  1120.49  2384.07   
Clothing        35.0    50.99   23.98    8.99   37.99    53.99    66.74   
Components     134.0   469.86  462.93   20.24  114.49   333.42   594.83   

                   max  
category_name           
Accessories     159.00  
Bikes          3578.27  
Clothing         89.99  
Components     1431.50  


In [17]:
chart = alt.Chart(priced).mark_boxplot(extent='min-max', size=40).encode(
    x=alt.X('category_name:N', title='Category',
            sort=['Bikes', 'Components', 'Clothing', 'Accessories']),
    y=alt.Y('list_price:Q',
            scale=alt.Scale(type='log', domain=[1, 5000]),
            title='List price (USD, log scale)'),
    color=alt.Color('category_name:N', legend=None)
).properties(
    title='List-price distribution per category (priced products)',
    width=500, height=350
)
chart.display()

alt.Chart(...)

In [18]:
# <Student to fill this section>
feature_2_insights = """
The four categories occupy four almost non-overlapping price regimes (note log scale):
- Bikes: USD 540 to USD 3,578 (median USD 1,120). The cheapest Bike is more expensive than the most expensive Clothing item.
- Components: USD 20 to USD 1,432 (median USD 333). The widest spread - includes both small parts (chains, brakes) and expensive frames.
- Clothing: USD 9 to USD 90 (median USD 54). Very tight range.
- Accessories: USD 2 to USD 159 (median USD 25). Widest relative spread on the low end.

The non-overlap is the key insight: a unit_price of USD 100 is normal in Components, expensive in Clothing, very expensive in Accessories and effectively impossible in Bikes. A single global model of "normal price" cannot capture this - the anomaly model MUST condition on category.

Modelling decision:
We will compute price-ratio and discount-ratio features at the category level (and where stable, at subcategory level). Specifically:
- price_vs_category_median = unit_price / category_median_list_price
- price_vs_listprice = unit_price / product.list_price
These ratios are scale-invariant and will let the Isolation Forest learn one consistent definition of "abnormal pricing" across all four categories.
"""

In [19]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `\<revenue and line_count contribution\>`

In [20]:
# <Student to fill this section>
# Join the full product chain to sales_order_detail to measure actual transactional weight
sod_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
sales_chain = chain.merge(sod_df[['product_id','order_quantity','line_total']],
                          on='product_id', how='inner')

cat_rev = sales_chain.groupby('category_name').agg(
    n_lines=('line_total','count'),
    total_revenue=('line_total','sum'),
    total_units=('order_quantity','sum'),
    avg_line_value=('line_total','mean'),
    median_line_value=('line_total','median')
).sort_values('total_revenue', ascending=False).round(2)
print("Transactional contribution per category:")
print(cat_rev)

print(f"\nBikes share of revenue: {cat_rev.loc['Bikes','total_revenue']/cat_rev['total_revenue'].sum()*100:.1f}%")
print(f"Accessories share of lines: {cat_rev.loc['Accessories','n_lines']/cat_rev['n_lines'].sum()*100:.1f}%")

Transactional contribution per category:
               n_lines  total_revenue  total_units  avg_line_value  \
category_name                                                        
Bikes            13299    27360878.47      21216.0         2057.36   
Components        3359     2124025.96       9004.0          632.34   
Clothing          6685      517485.42      17248.0           77.41   
Accessories      18757      490630.39      23973.0           26.16   

               median_line_value  
category_name                     
Bikes                    2071.42  
Components                291.58  
Clothing                   49.99  
Accessories                 9.99  

Bikes share of revenue: 89.7%
Accessories share of lines: 44.6%


In [21]:
plot_df = cat_rev.reset_index().assign(
    revenue_share = lambda d: d['total_revenue']/d['total_revenue'].sum()*100,
    line_share    = lambda d: d['n_lines']/d['n_lines'].sum()*100
)
long_df = plot_df[['category_name','revenue_share','line_share']].melt(
    id_vars='category_name', var_name='metric', value_name='share_pct')

chart = alt.Chart(long_df).mark_bar().encode(
    x=alt.X('category_name:N', sort=['Bikes','Components','Clothing','Accessories'],
            title='Category'),
    y=alt.Y('share_pct:Q', title='Share of total (%)'),
    color=alt.Color('metric:N',
                    scale=alt.Scale(domain=['revenue_share','line_share'],
                                    range=['#4C78A8','#F58518']),
                    title='Metric'),
    xOffset='metric:N',
    tooltip=['category_name','metric','share_pct']
).properties(
    title='Revenue share vs Line-count share by category',
    width=500, height=300
)
chart.display()

alt.Chart(...)

In [22]:
# <Student to fill this section>
feature_n_insights = """
The transactional weight of each category is the inverse of its intuition - the category with the fewest products and subcategories generates almost all the revenue:
- Bikes: 31.6% of all sales lines but 89.7% of total revenue (USD 27.4M of the USD 30.5M total). Average line value: USD 2,057.
- Accessories: 44.6% of lines but only 1.6% of revenue (USD 0.49M). Average line value: USD 26.
- Components: 8.0% of lines, 7.0% of revenue. Average line value: USD 632.
- Clothing: 15.9% of lines, 1.7% of revenue. Average line value: USD 77.

What this means for the anomaly model:
- A pricing or quantity error on a single Bikes line is ~80x more financially damaging than the same error on an Accessories line.
 The business cost of missing an anomaly is wildly different across categories, so when we evaluate the model we will report
 performance by category, not just globally.
- The model will see Accessories lines most frequently (44.6%) but the typical Accessories line carries almost no information about high-value pricing patterns. Without category-conditioning, the model would learn behaviour dominated by tiny, low-value lines and would be blind to large-value anomalies in Bikes.
- This justifies a deliberate feature-engineering choice: we will compute price/discount ratios AND we will keep the raw category as a feature so the model can learn category-specific thresholds. We may also experiment with training separate models per category in the experimentation phase.
"""

In [23]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)